# Projeto Python IA: Inteligência Artificial e Previsões

### Case: Score de Crédito dos Clientes

Você foi contratado por um banco para conseguir definir o score de crédito dos clientes. Você precisa analisar todos os clientes do banco e, com base nessa análise, criar um modelo que consiga ler as informações do cliente e dizer automaticamente o score de crédito dele: Ruim, Ok, Bom

Arquivos da aula: https://drive.google.com/drive/folders/1FbDqVq4XLvU85VBlVIMJ73p9oOu6u2-J?usp=drive_link

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# Passo 0 - Entender a empresa e o desafio da empresa
# Novo case: Banco de Fardos
# Objetivo: prever se cada fardo deve ser excluído, multiplicado ou dividido (ação)
# e também prever o valor associado (0 para excluir, ou número de multiplicação/divisão)

# Passo 1 - Importar a base de dados

# Substituir pelo caminho correto do arquivo
# Exemplo: banco_de_fardos.csv deve conter colunas como:
# id_fardo, peso, volume, tipo, material, destino, acao, valor

tabela = pd.read_csv("banco_de_fardos.csv")

display(tabela.head())

# Passo 2 - Preparar a base de dados
# Codificar colunas de texto (exceto a coluna alvo "acao")

codificadores = {}
for coluna in tabela.select_dtypes(include="object").columns:
    if coluna not in ["acao"]:
        codificadores[coluna] = LabelEncoder()
        tabela[coluna] = codificadores[coluna].fit_transform(tabela[coluna])

# y1 -> ação (excluir/multiplicar/dividir)
y1 = tabela["acao"]

# y2 -> valor (0, 2, 3, 0.5, etc)
y2 = tabela["valor"]

# X -> features
X = tabela.drop(columns=["acao", "valor", "id_fardo"])

# Passo 3 - Separar em treino e teste
X_treino, X_teste, y1_treino, y1_teste, y2_treino, y2_teste = train_test_split(
    X, y1, y2, test_size=0.3, random_state=42
)

# Passo 4 - Criar e treinar os modelos
modelo_acao = RandomForestClassifier()
modelo_valor = RandomForestClassifier()  # se valor for discreto
# Se for número contínuo usar:
# from sklearn.ensemble import RandomForestRegressor
# modelo_valor = RandomForestRegressor()

modelo_acao.fit(X_treino, y1_treino)
modelo_valor.fit(X_treino, y2_treino)

# Passo 5 - Avaliar a performance
print("Acurácia Modelo Ação:", modelo_acao.score(X_teste, y1_teste))
print("Acurácia Modelo Valor:", modelo_valor.score(X_teste, y2_teste))

# Passo 6 - Fazer previsão em novos fardos
# Arquivo de novos fardos deve ter a mesma estrutura (sem as colunas acao e valor)
novos_fardos = pd.read_csv("novos_fardos.csv")

# aplicar as mesmas transformações
for coluna, codificador in codificadores.items():
    novos_fardos[coluna] = codificador.transform(novos_fardos[coluna])

X_novos = novos_fardos.drop(columns=["id_fardo"])

previsao_acao = modelo_acao.predict(X_novos)
previsao_valor = modelo_valor.predict(X_novos)

# Juntar os resultados
resultado = novos_fardos[["id_fardo"]].copy()
resultado["acao"] = previsao_acao
resultado["valor"] = previsao_valor

display(resultado)
